### Demostración integrada: muestra real pequeña, recuperación y modelo real opcional

Este cuaderno ofrece una ruta corta para recorrer el proyecto sin pasar primero por toda la secuencia `00-06`.

#### Qué hace
- carga la muestra real pequeña ya normalizada,
- resume la composición del corpus,
- ejecuta una consulta de recuperación sobre el índice textual base,
- genera una salida grounded a partir de estructura, texto y vecinos recuperados,
- muestra dos casos comentados en profundidad,
- deja una celda opcional para correr un modelo real de captioning sobre una imagen local.

#### Qué no hace
- no sustituye los cuadernos `00-06`,
- no descarga imágenes patrimoniales de forma automática,
- no incluye una rúbrica de evaluación.

In [1]:
# Resuelve la raíz del proyecto de forma compatible con Docker/Linux.
from pathlib import Path
import sys
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io_utils import jsonl_load
from src.retrieval import fit_text_retriever, retrieve_neighbors_tfidf
from src.generation import grounded_prompt, grounded_generate_template
from src.runtime_env import collect_runtime_report

RECORDS_PATH = PROJECT_ROOT / "data_processed" / "records_master.jsonl"
records = jsonl_load(RECORDS_PATH)
df = pd.DataFrame(records)

print("Proyecto:", PROJECT_ROOT.name)
print("Directorio base:", PROJECT_ROOT)
print("Registros cargados:", len(df))
print(json.dumps(collect_runtime_report(PROJECT_ROOT), ensure_ascii=False, indent=2))

df[["record_id", "source", "modality_type", "title", "culture", "split"]]


Proyecto: Patrimonio_Andino_Grounded
Directorio base: /workspace/Semana6/Proyecto/Patrimonio_Andino_Grounded
Registros cargados: 7
{
  "cwd": "/workspace/Semana6/Proyecto/Patrimonio_Andino_Grounded/notebooks",
  "project_root": "/workspace/Semana6/Proyecto/Patrimonio_Andino_Grounded",
  "expected_root": "/workspace/Semana6/Proyecto/Patrimonio_Andino_Grounded",
  "in_workspace": "True",
  "records_master": "True",
  "jupyter_port": "8899",
  "python_executable": "",
  "torch_version": "2.4.1+cu121",
  "cuda_available": "True",
  "cuda_device_count": "1",
  "cuda_device_name_0": "NVIDIA GeForce RTX 4080 SUPER"
}


,record_id,source,modality_type,title,culture,split
0,okr_kh_0068_view_01,open_khipu,no_image,"Khipu KH0068 (UR1057, AS057)",Andean,train
1,okr_kh_0082_view_01,open_khipu,no_image,Khipu KH0082 (AS069),Andean,train
2,okr_kh_0323_view_01,open_khipu,no_image,Khipu KH0323 (UR087),Andean,dev
3,okr_kh_0328_view_01,open_khipu,no_image,Khipu KH0328 (UR092),Andean,train
4,par_1932_01_0005_photo_01,paracas,photo,Fragmento textil Paracas 1932.01.0005,Paracas,train
5,par_1935_32_0212_photo_01,paracas,photo,Fragmentos de manto Paracas 1935.32.0212,Paracas,test
6,par_xrf_em1932_01_0013e,paracas,xrf_map,Mapa XRF de muestra textil Paracas EM1932.01.0...,Paracas,challenge


#### **1. Composición de la muestra**

La muestra combina dos lógicas distintas:

- **Open Khipu**, donde domina la estructura y la descripción visual directa suele ser insuficiente,
- **Paracas**, donde conviene distinguir entre **foto patrimonial** e **imagen técnica**.

La separación por modalidad evita pedirle al sistema el mismo tipo de texto para evidencia visual que no cumple la misma función.

In [2]:
summary = (
    df.groupby(["source", "modality_type"])
      .size()
      .reset_index(name="n")
      .sort_values(["source", "modality_type"])
)
summary

,source,modality_type,n
0,open_khipu,no_image,4
1,paracas,photo,2
2,paracas,xrf_map,1


#### **2. Consulta de recuperación**

La recuperación base usa TF-IDF sobre título, descripción, tipo de objeto, cultura, procedencia y etiquetas.  
Aquí se puede observar si la consulta trae vecinos útiles antes de pasar a la generación.

In [3]:
retriever = fit_text_retriever(records)

query = "khipu andino con muchos cordeles y jerarquía compleja"
neighbors = retrieve_neighbors_tfidf(query, retriever, top_k=5)
pd.DataFrame(neighbors)

,record_id,canonical_id,title,source,score,description
0,okr_kh_0068_view_01,kh_0068,"Khipu KH0068 (UR1057, AS057)",open_khipu,0.176771,Khipu registrado en el Museo Regional de Ica A...
1,okr_kh_0328_view_01,kh_0328,Khipu KH0328 (UR092),open_khipu,0.174987,"Khipu del conjunto de Santa, conservado en el ..."
2,okr_kh_0323_view_01,kh_0323,Khipu KH0323 (UR087),open_khipu,0.163528,"Khipu del conjunto de Santa, conservado en el ..."
3,okr_kh_0082_view_01,kh_0082,Khipu KH0082 (AS069),open_khipu,0.154776,Khipu grande de colección privada Percy Dauels...
4,par_1932_01_0005_photo_01,par_1932_01_0005,Fragmento textil Paracas 1932.01.0005,paracas,0.016511,Fragmento arqueológico textil Paracas de estil...


#### **3. Generación grounded sobre un registro real**

La salida combina:
- baseline descriptivo,
- estructura (`x_s`),
- texto catalográfico (`x_t`),
- contexto (`x_c`),
- plausibilidad operativa,
- vecinos recuperados.

El objetivo no es producir una prosa brillante, sino una salida disciplinada y apoyada en evidencia.

In [4]:
record = next(r for r in records if r["record_id"] == "okr_kh_0082_view_01")

print(grounded_prompt(record, retriever, top_k=2)[:2200])

Usa solo la evidencia disponible.
No inventes cronología, procedencia, materialidad ni significado.
Separa observación, metadatos e inferencia.
Explicita incertidumbre.

[OBJETO]
id: okr_kh_0082_view_01
title: Khipu KH0082 (AS069)
object_type: Khipu

[BASELINE]
Khipu con 1831 cordeles, 46 colores registrados y procedencia Lluta Valley.

[ESTRUCTURA]
{"material": "fibra textil", "cord_count": 1831, "pendant_count": 1650, "top_cord": null, "knot_types": [], "spin_ply": null, "color_terms": [], "hierarchy": "458 grupos", "attachment_structure": null, "unique_colors": 46}

[TEXTO]
{"title": "Khipu KH0082 (AS069)", "description": "Private collection Percy Dauelsberg, Arica; provenance Lluta Valley; 1831 cords.", "keywords": ["Percy Dauelsberg", "Lluta Valley", "46 colors"]}

[CONTEXTO]
{"chronology": null, "geography": "Arica, Chile", "museum": "Private collection, Percy Dauelsberg (current location unknown)", "museum_number": "4", "region": "Unknown", "reference_url": "https://www.khipufie

In [5]:
output = grounded_generate_template(record, retriever, top_k=2)
output

{'caption_factual': 'Khipu con 1831 cordeles, 46 colores registrados y procedencia Lluta Valley.',
 'nota_tecnico_curatorial': 'Khipu KH0082 (AS069) se documenta como khipu asociado a Andean. La ficha disponible sitúa la procedencia en Lluta Valley y aporta como soporte material fibra textil anudada. Los atributos estructurales disponibles incluyen patrón=None, material=fibra textil, análisis=None, jerarquía=458 grupos, cordeles=1831.',
 'nota_comparativa': 'La recuperación sugiere afinidad parcial con Khipu KH0068 (UR1057, AS057); Khipu KH0328 (UR092). Esa proximidad es útil para comparación curatorial, pero no demuestra identidad de origen, función o cronología.',
 'incertidumbre': 'La evidencia actual no permite afirmar una lectura funcional o simbólica cerrada, y cualquier interpretación debe mantenerse provisional.',
 'traza_evidencia': ['baseline: Khipu con 1831 cordeles, 46 colores registrados y procedencia Lluta Valley.',
  'provenance: Lluta Valley',
  'material: fibra textil 

#### **4. Dos casos comentados en profundidad**

En esta versión, el proyecto ya trae dos casos con comentario narrativo más extenso:
- `KH0082` para la lógica estructural de Open Khipu,
- `1935.32.0212` para la lógica curatorial de Paracas.

La idea es contrastar un caso donde predomina la estructura con otro donde pesan más los metadatos curatoriales y la comparación.

In [ ]:
cases_dir = PROJECT_ROOT / "docs" / "casos_comentados"
for path in sorted(cases_dir.glob("*.md")):
    print("=" * 90)
    print(path.name)
    print("=" * 90)
    print(path.read_text()[:2500])
    print()

#### **5. Ruta opcional con un modelo real de captioning**

Esta celda se deja desactivada por defecto.  
Requiere una imagen local y dependencias de `transformers`, `torch` y `Pillow`.

La recomendación es usarla solo como extensión, por ejemplo con:
- una fotografía permitida del proyecto,
- una imagen pública local ya descargada por el docente,
- o una imagen técnica de prueba que sí pueda redistribuirse.

La salida se puede comparar luego con la versión grounded para discutir qué agrega la evidencia estructural.

In [6]:
RUN_REAL_CAPTIONING = False
LOCAL_IMAGE_PATH = PROJECT_ROOT / "data_raw" / "demo_real_image.jpg"

if RUN_REAL_CAPTIONING:
    from PIL import Image
    import torch
    from transformers import BlipProcessor, BlipForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

    image = Image.open(LOCAL_IMAGE_PATH).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    generated_ids = model.generate(**inputs, max_new_tokens=40)
    caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    print("Caption real:", caption)
else:
    print("Ruta opcional desactivada. Cambia RUN_REAL_CAPTIONING a True y coloca una imagen local en data_raw/demo_real_image.jpg")

Ruta opcional desactivada. Cambia RUN_REAL_CAPTIONING a True y coloca una imagen local en data_raw/demo_real_image.jpg


#### **6. Próximo paso recomendado**

Después de revisar este cuaderno, la secuencia sugerida es:

1. `03_recuperacion_base_e_indices.ipynb`
2. `04_generacion_grounded.ipynb`
3. `06_casos_curatoriales_y_exportacion.ipynb`
